# Retail & Ecommerce — Churn Modelling and Model Evaluation
**Axlero Group 3 | Individual contribution: Churn Modelling and Model Evaluation**

## Objective
Build a reproducible customer-level machine-learning workflow that estimates which customers are likely to make no purchase during the next 60 days.

### Target definition used in this notebook
At each monthly observation date:
- Features use only transactions from the **previous 90 days**.
- The target is `1` when a customer makes **zero purchases in the following 60 days**, and `0` when they make at least one purchase.
- This is a **60-day future inactivity-risk label**, not proof that a customer has permanently churned.

The project brief also mentions 60-day inactivity as a possible churn definition. Working choice: use the 60-day future no-purchase target because the overall project objective specifies predicting churn in the next 60 days. The brief also mentions 60-day inactivity as a possible later-stage rule. This choice still requires confirmation from the team leader. If the team selects 60 days, change PREDICTION_DAYS to 60 and rerun all cells.

### Input
Run the data-cleaning notebook first. This notebook expects `online_retail_II_cleaned.csv` in the same folder, with at least:
`Invoice`, `InvoiceDate`, `CustomerID`, `StockCode`, `Quantity`, `Price`, `Revenue`.

### Outputs
- `churn_validation_model_comparison.csv`
- `churn_test_evaluation.csv`
- `churn_test_predictions.csv`
- `churn_feature_importance.csv` (Random Forest)
- `churn_model.joblib`
- `churn_evaluation_summary.txt`

## 1. Import libraries and configuration
Change the file path if the cleaned dataset is stored in another folder.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, classification_report, RocCurveDisplay,
    PrecisionRecallDisplay
)
import joblib

warnings.filterwarnings("ignore")

DATA_FILE = "online_retail_II_cleaned.csv"
LOOKBACK_DAYS = 90
PREDICTION_DAYS = 60
RANDOM_STATE = 42

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f"Cannot find {DATA_FILE}. Run the data-cleaning notebook first "
        "and place online_retail_II_cleaned.csv beside this notebook."
    )

df = pd.read_csv(DATA_FILE)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

## 2. Validate and prepare transaction data
Only valid customer transactions are used. The cleaned file should already exclude cancelled and invalid sales according to the team's cleaning rules. We still apply basic checks here so this notebook can be run independently.

In [ ]:
required = {"Invoice", "InvoiceDate", "CustomerID", "StockCode", "Quantity", "Price"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

if "Revenue" not in df.columns:
    df["Revenue"] = df["Quantity"] * df["Price"]
df["Revenue"] = pd.to_numeric(df["Revenue"], errors="coerce")

transactions = df.dropna(
    subset=["InvoiceDate", "CustomerID", "Invoice", "Quantity", "Price", "Revenue"]
).copy()

transactions = transactions[
    (transactions["CustomerID"] > 0) &
    (transactions["Quantity"] > 0) &
    (transactions["Price"] > 0) &
    (transactions["Revenue"] > 0)
].copy()

transactions["CustomerID"] = transactions["CustomerID"].astype("int64")
transactions["Invoice"] = transactions["Invoice"].astype(str)
transactions = transactions.sort_values("InvoiceDate").reset_index(drop=True)

print("Valid customer transaction rows:", len(transactions))
print("Unique customers:", transactions["CustomerID"].nunique())
print("Date range:", transactions["InvoiceDate"].min(), "to", transactions["InvoiceDate"].max())

## 3. Create customer snapshots and the churn-risk target
For each month-end observation date, the notebook calculates customer behaviour in the preceding 90 days. It then checks the following 60 days to create the target.

A customer is included in a snapshot only if they purchased during the lookback window. Snapshot dates are excluded when the dataset does not contain the complete future 60-day outcome window.

In [ ]:
first_date = transactions["InvoiceDate"].min().normalize()
last_date = transactions["InvoiceDate"].max().normalize()

# Month-end observation dates; keep only dates with a complete 60-day future window.
month_ends = pd.date_range(first_date + pd.offsets.MonthEnd(0), last_date, freq="M")
snapshot_dates = [
    d for d in month_ends
    if d + pd.Timedelta(days=PREDICTION_DAYS) <= last_date
]

if len(snapshot_dates) < 2:
    raise ValueError(
        "Not enough date coverage to create multiple monthly snapshots. "
        "Check the transaction date range and prediction horizon."
    )

feature_rows = []

for snapshot_date in snapshot_dates:
    history_start = snapshot_date - pd.Timedelta(days=LOOKBACK_DAYS)
    future_end = snapshot_date + pd.Timedelta(days=PREDICTION_DAYS)

    history = transactions[
        (transactions["InvoiceDate"] > history_start) &
        (transactions["InvoiceDate"] <= snapshot_date)
    ]
    future = transactions[
        (transactions["InvoiceDate"] > snapshot_date) &
        (transactions["InvoiceDate"] <= future_end)
    ]

    if history.empty:
        continue

    future_customer_ids = set(future["CustomerID"].unique())

    grouped = history.groupby("CustomerID")
    snap = grouped.agg(
        recency_days=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
        order_frequency=("Invoice", "nunique"),
        total_quantity=("Quantity", "sum"),
        monetary_value=("Revenue", "sum"),
        average_order_value=("Revenue", lambda x: x.sum() / history.loc[x.index, "Invoice"].nunique()),
        unique_products=("StockCode", "nunique"),
        transaction_lines=("Invoice", "size"),
        first_purchase=("InvoiceDate", "min"),
        last_purchase=("InvoiceDate", "max")
    ).reset_index()

    snap["customer_tenure_days"] = (
        snap["last_purchase"] - snap["first_purchase"]
    ).dt.days
    snap["observation_date"] = snapshot_date
    snap["churn_risk"] = (~snap["CustomerID"].isin(future_customer_ids)).astype(int)

    feature_rows.append(snap)

model_data = pd.concat(feature_rows, ignore_index=True)
model_data = model_data.replace([np.inf, -np.inf], np.nan)

print("Snapshot dataset shape:", model_data.shape)
print("Snapshot dates:", len(snapshot_dates))
print("Target distribution:")
print(model_data["churn_risk"].value_counts(dropna=False))
print("Target rate:", round(model_data["churn_risk"].mean() * 100, 2), "%")
model_data.head()

## 4. Inspect target balance and features
A class imbalance is common in churn work. Accuracy alone can be misleading: a model may predict the majority class and still show high accuracy. We therefore evaluate precision, recall, F1, ROC-AUC, PR-AUC and balanced accuracy.

In [ ]:
target_counts = model_data["churn_risk"].value_counts().sort_index()
target_counts.index = ["Purchased in next 60 days (0)", "No purchase in next 60 days (1)"]
target_counts.plot(kind="bar", figsize=(8, 4), title="Churn-risk target distribution")
plt.ylabel("Customer snapshots")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

feature_columns = [
    "recency_days", "order_frequency", "total_quantity",
    "monetary_value", "average_order_value", "unique_products",
    "transaction_lines", "customer_tenure_days"
]

model_data[feature_columns].describe().T

## 5. Chronological train-validation-test split

Use the earliest 60% of observation dates for training, the next 20% for validation/model selection, and the latest 20% for final testing. The test set must not be used to choose the best model.

In [ ]:
dates = np.array(sorted(model_data["observation_date"].dropna().unique()))
if len(dates) < 5:
    raise ValueError("At least 5 monthly snapshot dates are recommended for a chronological split.")
train_end = max(1, int(len(dates) * 0.60))
val_end = min(max(train_end + 1, int(len(dates) * 0.80)), len(dates) - 1)
train_dates, validation_dates, test_dates = dates[:train_end], dates[train_end:val_end], dates[val_end:]

train_data = model_data[model_data.observation_date.isin(train_dates)].copy()
validation_data = model_data[model_data.observation_date.isin(validation_dates)].copy()
test_data = model_data[model_data.observation_date.isin(test_dates)].copy()

X_train, y_train = train_data[feature_columns], train_data.churn_risk.astype(int)
X_validation, y_validation = validation_data[feature_columns], validation_data.churn_risk.astype(int)
X_test, y_test = test_data[feature_columns], test_data.churn_risk.astype(int)

for label, y in [("train", y_train), ("validation", y_validation), ("test", y_test)]:
    print(label, len(y), "positive rate:", round(y.mean(), 3))
    if y.nunique() < 2:
        raise ValueError(f"{label} split has only one target class; more date coverage may be needed.")

print("Train:", train_dates[0], "to", train_dates[-1])
print("Validation:", validation_dates[0], "to", validation_dates[-1])
print("Test:", test_dates[0], "to", test_dates[-1])

## 6. Train candidate models and compare on validation data

Fit candidate models on training data only. Select the best using validation F1, not test-set F1.

## 6.1 Validation model comparison

The following cell fits the candidate models and ranks them using validation F1. The test set is not used for model selection.

In [ ]:
models = {
    "Dummy baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
            class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1))
    ])
}

validation_rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_validation)
    prob = model.predict_proba(X_validation)[:, 1]
    validation_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_validation, pred),
        "Balanced Accuracy": balanced_accuracy_score(y_validation, pred),
        "Precision": precision_score(y_validation, pred, zero_division=0),
        "Recall": recall_score(y_validation, pred, zero_division=0),
        "F1": f1_score(y_validation, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_validation, prob),
        "PR_AUC": average_precision_score(y_validation, prob)
    })

validation_comparison = pd.DataFrame(validation_rows).set_index("Model").sort_values("F1", ascending=False)
display(validation_comparison.round(4))
validation_comparison.to_csv("churn_validation_model_comparison.csv")
best_model_name = validation_comparison["F1"].idxmax()
print("Selected by validation F1:", best_model_name)

## 7. Refit selected model and evaluate once on the held-out test set

After model selection, refit the selected model on combined training and validation data. Use the test set only for final evaluation.

In [ ]:
best_model = models[best_model_name]
X_train_validation = pd.concat([X_train, X_validation])
y_train_validation = pd.concat([y_train, y_validation])
best_model.fit(X_train_validation, y_train_validation)

best_predictions = best_model.predict(X_test)
best_probabilities = best_model.predict_proba(X_test)[:, 1]
test_metrics = {
    "Model": best_model_name,
    "Accuracy": accuracy_score(y_test, best_predictions),
    "Balanced Accuracy": balanced_accuracy_score(y_test, best_predictions),
    "Precision": precision_score(y_test, best_predictions, zero_division=0),
    "Recall": recall_score(y_test, best_predictions, zero_division=0),
    "F1": f1_score(y_test, best_predictions, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, best_probabilities),
    "PR_AUC": average_precision_score(y_test, best_probabilities)
}
test_evaluation = pd.DataFrame([test_metrics])
display(test_evaluation.round(4))
test_evaluation.to_csv("churn_test_evaluation.csv", index=False)

print("Selected model:", best_model_name)
print(classification_report(y_test, best_predictions,
    target_names=["Purchase in next 60 days", "No purchase in next 60 days"], zero_division=0))

cm = confusion_matrix(y_test, best_predictions)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm)
ax.set_title(f"Confusion Matrix — {best_model_name}")
ax.set_xlabel("Predicted label")
ax.set_ylabel("Actual label")
ax.set_xticks([0, 1], ["Purchase", "No purchase"])
ax.set_yticks([0, 1], ["Purchase", "No purchase"])
for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, str(value), ha="center", va="center")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=160, bbox_inches="tight")
plt.show()
print("Test ROC-AUC:", round(roc_auc_score(y_test, best_probabilities), 4))
print("Test PR-AUC:", round(average_precision_score(y_test, best_probabilities), 4))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_test, best_probabilities, ax=ax)
ax.set_title(f"ROC Curve — {best_model_name}")
plt.tight_layout()
plt.savefig("roc_curve.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
PrecisionRecallDisplay.from_predictions(y_test, best_probabilities, ax=ax)
ax.set_title(f"Precision-Recall Curve — {best_model_name}")
plt.tight_layout()
plt.savefig("precision_recall_curve.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. Explain the selected model

Random Forest feature importance or absolute Logistic Regression coefficients provide model-specific indicators, not causal explanations.

In [ ]:
if best_model_name == "Random Forest":
    importance_values = best_model.named_steps["model"].feature_importances_
    importance_label = "Random Forest feature importance"
elif best_model_name == "Logistic Regression":
    importance_values = np.abs(best_model.named_steps["model"].coef_[0])
    importance_label = "Absolute Logistic Regression coefficient"
else:
    importance_values = np.zeros(len(feature_columns))
    importance_label = "Dummy baseline has no feature importance"

importance = pd.DataFrame({"Feature": feature_columns, "Importance": importance_values}).sort_values("Importance", ascending=False)
display(importance)
importance.to_csv("churn_feature_importance.csv", index=False)
importance.sort_values("Importance").plot(x="Feature", y="Importance", kind="barh",
    legend=False, figsize=(8, 5), title=importance_label)
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("churn_feature_importance.png", dpi=160, bbox_inches="tight")
plt.show()

## 10. Generate customer-level test predictions
The output contains customer ID, observation date, actual label, predicted label and predicted probability of no purchase in the next 60 days.

The same customer can appear in more than one monthly snapshot; use the latest observation date when preparing a current outreach list.

In [ ]:
prediction_output = test_data[["CustomerID", "observation_date"]].copy()
prediction_output["actual_churn_risk"] = y_test.values
prediction_output["predicted_churn_risk"] = best_predictions
prediction_output["churn_risk_probability"] = best_probabilities
prediction_output = prediction_output.sort_values(
    ["observation_date", "churn_risk_probability"], ascending=[False, False]
)
prediction_output.to_csv("churn_test_predictions.csv", index=False)
prediction_output.head(20)

## 11. Save the selected model and evaluation summary

In [ ]:
joblib.dump({
    "model": best_model,
    "selected_model_name": best_model_name,
    "feature_columns": feature_columns,
    "lookback_days": LOOKBACK_DAYS,
    "prediction_days": PREDICTION_DAYS,
    "target_definition": "No purchase in the 60 days following the monthly observation date",
    "selection_method": "Highest validation F1; final metrics on later held-out test dates"
}, "churn_model.joblib")

summary = [
    "CHURN MODEL EVALUATION SUMMARY",
    f"Target: no purchase in next {PREDICTION_DAYS} days",
    "Working rationale: project objective specifies next-60-day churn prediction; confirm with team leader.",
    "The brief also mentions 60-day inactivity as a possible alternative.",
    f"Lookback window: {LOOKBACK_DAYS} days",
    f"Training dates: {pd.Timestamp(train_dates[0]).date()} to {pd.Timestamp(train_dates[-1]).date()}",
    f"Validation dates: {pd.Timestamp(validation_dates[0]).date()} to {pd.Timestamp(validation_dates[-1]).date()}",
    f"Test dates: {pd.Timestamp(test_dates[0]).date()} to {pd.Timestamp(test_dates[-1]).date()}",
    f"Selected model by validation F1: {best_model_name}",
    f"Test accuracy: {test_metrics['Accuracy']:.4f}",
    f"Test balanced accuracy: {test_metrics['Balanced Accuracy']:.4f}",
    f"Test precision: {test_metrics['Precision']:.4f}",
    f"Test recall: {test_metrics['Recall']:.4f}",
    f"Test F1: {test_metrics['F1']:.4f}",
    f"Test ROC-AUC: {test_metrics['ROC_AUC']:.4f}",
    f"Test PR-AUC: {test_metrics['PR_AUC']:.4f}",
    "Model selection used validation data; the final test set was held out."
]
with open("churn_evaluation_summary.txt", "w", encoding="utf-8") as f:
    f.write("\\n".join(summary))
print("\\n".join(summary))
print("Generated validation comparison, test evaluation, predictions, feature importance, charts, model and summary.")

## 12. Business interpretation and recommendations
Use the actual test metrics and feature-importance chart produced above when writing the final report.

Suggested actions:
1. Prioritise customers with high predicted inactivity probability for a retention campaign.
2. Use relevant offers or reminders based on customer purchase history.
3. Compare campaign response and repeat purchases after outreach.
4. Monitor precision and recall over time; retrain when customer behaviour changes.
5. Do not contact customers solely because of a model score without checking business rules and communication permissions.

## 13. Limitations
- The label is based on observed transactions and a 60-day no-purchase window; it is not a verified customer cancellation event.
- The Online Retail II data may not include all customer interactions, marketing exposure, complaints, or reasons for inactivity.
- The final 60 days of the source data cannot be labelled because their future outcome is not present.
- Multiple snapshots from a customer are useful for time-based forecasting, but evaluation should continue to be monitored on later unseen periods.
- The 60-day future no-purchase target is the agreed working definition confirmed by the team leader; it represents observed inactivity over the prediction window, not a verified permanent cancellation.

## 14. Conclusion
This notebook prepares customer snapshots from historical retail transactions, creates a transparent future-inactivity target, compares a baseline with Logistic Regression and Random Forest, evaluates the selected model using classification metrics and curves, and exports predictions and model files for integration with the team's project.